# 07 — CNN Image Fundamentals

A compact convolutional neural network from scratch in PyTorch. The synthetic images contain either a vertical or horizontal bar, so the notebook focuses on what convolution, channels, pooling and flattening actually do rather than dataset downloading.

In [ ]:
import numpy as np
import torch
from torch import nn

torch.manual_seed(42)
np.random.seed(42)

def make_image(label, size=12):
    image = np.random.normal(0, 0.08, (size, size)).astype('float32')
    if label == 0:  # vertical bar
        image[:, 5:7] += 1.0
    else:           # horizontal bar
        image[5:7, :] += 1.0
    return image

labels = np.array([0, 1] * 500)
images = np.stack([make_image(label) for label in labels])
X = torch.tensor(images).unsqueeze(1)  # [batch, channel, height, width]
y = torch.tensor(labels, dtype=torch.long)
perm = torch.randperm(len(X))
cut = int(0.8 * len(X))
train_idx, test_idx = perm[:cut], perm[cut:]
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]
print(X_train.shape)

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),             # 12x12 -> 6x6
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),             # 6x6 -> 3x3
        )
        self.classifier = nn.Linear(16 * 3 * 3, 2)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)

model = TinyCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
model

In [ ]:
for epoch in range(61):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 15 == 0:
        print(f'epoch={epoch:2d} loss={loss.item():.4f}')

In [ ]:
model.eval()
with torch.no_grad():
    pred = model(X_test).argmax(dim=1)
    accuracy = (pred == y_test).float().mean().item()
print(f'test accuracy: {accuracy:.3f}')

## CNN building blocks

- **Convolution** learns local filters that can respond to edges, textures and later more complex patterns.
- **Channels** are separate learned feature maps; the first convolution maps one input channel to eight learned channels.
- **ReLU** adds non-linearity.
- **Pooling** reduces spatial resolution and computation while keeping strong local responses.
- **Flatten + linear layer** turns the final feature maps into class logits.

The full portfolio image-classification project builds on these basics with transfer learning, uncertainty, Grad-CAM and model export.